# Bluff Poker Monte Carlo Simulator (PokerKit, non-interactive)

Requires:
    pip install pokerkit

Usage (e.g. in Jupyter):
  1. Edit the CONFIG section (NUM_PLAYERS, HERO_CARD_STRINGS, CLAIM_TYPE, SIMULATIONS).
  2. Run this cell.
  3. It will print the estimated % that the claim is true vs a bluff.

Model:
  - NUM_PLAYERS players total, each with 2 cards.
  - HERO (you) always has HERO_CARD_STRINGS (fixed across simulations).
  - In each simulation we:
      * Build a full deck, remove hero's cards.
      * Deal 2 cards to every other player.
      * Pool ALL 2 * NUM_PLAYERS cards.
      * Check if the CLAIM_TYPE hand exists anywhere in that pool.
  

In [1]:
%pip install pokerkit
import random
from collections import Counter
from typing import List

from pokerkit import Card
from pokerkit.utilities import Rank, Suit, Deck


# ====== CONFIG (edit these) =====================================

NUM_PLAYERS = 4                    # total players at the table (2–9 reasonable)
HERO_CARD_STRINGS = ["As", "Kd"]   # your two cards, e.g. ["As", "Kd"]
CLAIM_TYPE = "flush"               # e.g. "pair", "two pair", "straight", "flush", ...
SIMULATIONS = 1000                 # how many random deals to simulate


# ====== RANK helpers for straights ==============================

RANK_ORDER = "23456789TJQKA"
RANK_TO_VALUE = {ch: val for val, ch in enumerate(RANK_ORDER, start=2)}


# ====== PokerKit helper functions ===============================

def build_deck() -> List[Card]:
    """Return a mutable list copy of the standard 52-card deck."""
    return list(Deck.STANDARD)


def parse_card(token: str) -> Card:
    """
    Parse a 2-character card like 'As' or 'Td' into a PokerKit Card.
    Ranks: 2-9, T, J, Q, K, A
    Suits: c, d, h, s
    """
    token = token.strip()
    if len(token) != 2:
        raise ValueError(f"Bad card '{token}'. Use e.g. As, Td, 9h.")

    rank_char = token[0].upper()
    suit_char = token[1].lower()

    try:
        rank = Rank(rank_char)
        suit = Suit(suit_char)
    except ValueError as e:
        raise ValueError(
            f"Bad card '{token}'. "
            "Valid ranks: 2-9,T,J,Q,K,A; suits: c,d,h,s."
        ) from e

    return Card(rank, suit)


def parse_hero_cards(card_strings: List[str]) -> List[Card]:
    if len(card_strings) != 2:
        raise ValueError("Hero must have exactly two cards.")
    cards = [parse_card(s) for s in card_strings]
    if cards[0] == cards[1]:
        raise ValueError("Hero cards must be different.")
    return cards


# ====== Hand property helpers (on combined cards) ===============

def _extract_ranks_suits(cards: List[Card]):
    # We assume Rank.value is like '2','3',...,'A' and Suit.value 'c','d','h','s'
    ranks = [c.rank.value for c in cards]
    suits = [c.suit.value for c in cards]
    return ranks, suits


def _has_pair(rank_counts: Counter) -> bool:
    return any(cnt >= 2 for cnt in rank_counts.values())


def _has_two_pair(rank_counts: Counter) -> bool:
    # Two different ranks with count >= 2
    return sum(1 for cnt in rank_counts.values() if cnt >= 2) >= 2


def _has_three_of_a_kind(rank_counts: Counter) -> bool:
    return any(cnt >= 3 for cnt in rank_counts.values())


def _has_four_of_a_kind(rank_counts: Counter) -> bool:
    return any(cnt >= 4 for cnt in rank_counts.values())


def _has_full_house(rank_counts: Counter) -> bool:
    triples = [r for r, c in rank_counts.items() if c >= 3]
    pairs = [r for r, c in rank_counts.items() if c >= 2]
    for t in triples:
        for p in pairs:
            if p != t:
                return True
    return False


def _has_flush(suit_counts: Counter) -> bool:
    return any(cnt >= 5 for cnt in suit_counts.values())


def _has_straight(ranks: List[str]) -> bool:
    # convert rank chars to numeric values
    values = sorted({RANK_TO_VALUE[r] for r in ranks})
    if len(values) < 5:
        return False

    # Normal high straight (e.g., 9-T-J-Q-K)
    run = 1
    for i in range(1, len(values)):
        if values[i] == values[i - 1] + 1:
            run += 1
            if run >= 5:
                return True
        elif values[i] != values[i - 1]:
            run = 1

    # Wheel straight A-2-3-4-5 (treat Ace as 1)
    if 14 in values:
        wheel_vals = [1 if v == 14 else v for v in values]
        wheel_vals = sorted(set(wheel_vals))
        run = 1
        for i in range(1, len(wheel_vals)):
            if wheel_vals[i] == wheel_vals[i - 1] + 1:
                run += 1
                if run >= 5:
                    return True
            elif wheel_vals[i] != wheel_vals[i - 1]:
                run = 1

    return False


def _has_straight_flush(cards: List[Card], suit_counts: Counter) -> bool:
    # Any straight among cards of a single suit
    for suit, cnt in suit_counts.items():
        if cnt < 5:
            continue
        suited_ranks = [c.rank.value for c in cards if c.suit.value == suit]
        if _has_straight(suited_ranks):
            return True
    return False


def claim_satisfied(cards: List[Card], claim_type: str) -> bool:
    """
    Does the CLAIMED hand exist anywhere in these cards?

    Important: stronger hands still “satisfy” weaker claims:
      - any 3/4-of-a-kind/full house also satisfies "pair"
      - straight flush counts as both a straight AND a flush
    """
    ranks, suits = _extract_ranks_suits(cards)
    rank_counts = Counter(ranks)
    suit_counts = Counter(suits)

    ct = claim_type.lower().strip()

    if ct == "high card":
        # With this many cards, you'll always have at least a high card.
        return True
    if ct in ("pair", "one pair"):
        return _has_pair(rank_counts)
    if ct in ("two pair", "two pairs"):
        return _has_two_pair(rank_counts)
    if ct in ("three of a kind", "trips", "set"):
        return _has_three_of_a_kind(rank_counts)
    if ct == "straight":
        return _has_straight(ranks)
    if ct == "flush":
        return _has_flush(suit_counts)
    if ct == "full house":
        return _has_full_house(rank_counts)
    if ct in ("four of a kind", "quads"):
        return _has_four_of_a_kind(rank_counts)
    if ct == "straight flush":
        return _has_straight_flush(cards, suit_counts)

    raise ValueError(f"Unknown claim type: {claim_type!r}")


# ====== Simulation ==============================================

def simulate_claim(
    num_players: int,
    hero_cards: List[Card],
    claim_type: str,
    simulations: int = 1000,
) -> tuple[float, float]:
    """
    Simulate full deals with num_players, all 2-card hands turned up at the end.

    At showdown, all 2 * num_players cards are pooled and we check whether the
    claimed hand exists somewhere in that combined set.

    Returns:
        (true_percentage, bluff_percentage)
    """
    if num_players < 2:
        raise ValueError("Need at least 2 players.")
    if num_players * 2 > 52:
        raise ValueError("Too many players for a 52-card deck.")

    true_count = 0

    for _ in range(simulations):
        deck = build_deck()

        # Remove hero's actual cards from the deck.
        for c in hero_cards:
            deck.remove(c)

        random.shuffle(deck)

        # Deal 2 cards to each opponent.
        all_cards = list(hero_cards)
        for _ in range(num_players - 1):
            all_cards.append(deck.pop())
            all_cards.append(deck.pop())

        if claim_satisfied(all_cards, claim_type):
            true_count += 1

    true_pct = 100.0 * true_count / simulations
    bluff_pct = 100.0 - true_pct
    return true_pct, bluff_pct


# ====== RUN SIMULATION (top-level) ==============================

if __name__ == "__main__" or True:
    hero_cards = parse_hero_cards(HERO_CARD_STRINGS)
    true_pct, bluff_pct = simulate_claim(
        num_players=NUM_PLAYERS,
        hero_cards=hero_cards,
        claim_type=CLAIM_TYPE,
        simulations=SIMULATIONS,
    )

    print("=== Bluff Poker Simulator (PokerKit, non-interactive) ===")
    print(f"Players at table:     {NUM_PLAYERS}")
    print(f"Hero cards:           {HERO_CARD_STRINGS}")
    print(f"Claimed hand type:    {CLAIM_TYPE!r}")
    print(f"Simulations run:      {SIMULATIONS}")
    print()
    print(f"Claim satisfied:      {true_pct:.1f}% of deals")
    print(f"Estimated bluff rate: {bluff_pct:.1f}% of deals")


Note: you may need to restart the kernel to use updated packages.
=== Bluff Poker Simulator (PokerKit, non-interactive) ===
Players at table:     4
Hero cards:           ['As', 'Kd']
Claimed hand type:    'flush'
Simulations run:      1000

Claim satisfied:      5.5% of deals
Estimated bluff rate: 94.5% of deals
